In [1]:
import os

print("Datasets mounted at /kaggle/input:")
print(os.listdir("/kaggle/input"))
print(os.listdir("/kaggle/input/datasets")[:50])
print(os.listdir("/kaggle/input/datasets/organizations")[:100])

Datasets mounted at /kaggle/input:
['datasets']
['organizations']
['nih-chest-xrays']


In [2]:
import os
DATA_ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays"
print(os.listdir(DATA_ROOT)[:200])

['data']


In [3]:
import os
DATA_ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"

first_png = None
for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        if f.endswith(".png"):
            first_png = os.path.join(root, f)
            break
    if first_png:
        break

print("First PNG found:", first_png)


First PNG found: /kaggle/input/datasets/organizations/nih-chest-xrays/data/images_003/images/00006199_010.png


In [4]:
!nvidia-smi

Fri Mar 27 03:25:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:

import os, random
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


from torchvision import transforms
from sklearn.metrics import roc_auc_score

In [6]:
DATA_ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"
CSV_PATH = DATA_ROOT + "/Data_Entry_2017.csv"

LABELS = [
    "Atelectasis","Cardiomegaly","Effusion","Infiltration","Mass","Nodule",
    "Pneumonia","Pneumothorax","Consolidation","Edema","Emphysema","Fibrosis",
    "Pleural_Thickening","Hernia"
]
label2idx = {l:i for i,l in enumerate(LABELS)}
NUM_LABELS = len(LABELS)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [7]:
DATA_ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"
CSV_PATH = DATA_ROOT + "/Data_Entry_2017.csv"

LABELS = [
    "Atelectasis","Cardiomegaly","Effusion","Infiltration","Mass","Nodule",
    "Pneumonia","Pneumothorax","Consolidation","Edema","Emphysema","Fibrosis",
    "Pleural_Thickening","Hernia"
]
label2idx = {l:i for i,l in enumerate(LABELS)}
NUM_LABELS = len(LABELS)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [8]:
img_dirs = [
    os.path.join(DATA_ROOT, d, "images")
    for d in os.listdir(DATA_ROOT)
    if d.startswith("images_") and os.path.isdir(os.path.join(DATA_ROOT, d, "images"))
]

print("Image folders found:", len(img_dirs))
print("Example dirs:", img_dirs[:3])

name2path = {}
for d in img_dirs:
    for f in os.listdir(d):
        if f.endswith(".png"):
            name2path[f] = os.path.join(d, f)

print("Indexed images:", len(name2path))
print("Has 00000001_000.png?", "00000001_000.png" in name2path)

Image folders found: 12
Example dirs: ['/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_003/images', '/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_012/images', '/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_009/images']
Indexed images: 112120
Has 00000001_000.png? True


In [9]:
df = pd.read_csv(CSV_PATH)

patients = df["Patient ID"].unique()
np.random.shuffle(patients)

n = len(patients)
train_ids = set(patients[: int(0.8*n)])
val_ids   = set(patients[int(0.8*n): int(0.9*n)])
test_ids  = set(patients[int(0.9*n):])

train_df = df[df["Patient ID"].isin(train_ids)].copy()
val_df   = df[df["Patient ID"].isin(val_ids)].copy()
test_df  = df[df["Patient ID"].isin(test_ids)].copy()

print(len(train_df), len(val_df), len(test_df))

89703 11221 11196


In [10]:
class ChestXray14(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        fname = row["Image Index"]
        img_path = name2path[fname]  # <-- key fix

        img = Image.open(img_path).convert("L")

        y = np.zeros(NUM_LABELS, dtype=np.float32)
        for f in str(row["Finding Labels"]).split("|"):
            if f in label2idx:
                y[label2idx[f]] = 1.0

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(y)

IMG_SIZE = 224
train_tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

val_tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

test_tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

In [11]:
BATCH_SIZE = 32
NUM_WORKERS = 0

train_loader = DataLoader(ChestXray14(train_df, train_tf),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)

val_loader = DataLoader(ChestXray14(val_df, val_tf),
                        batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

test_loader = DataLoader(ChestXray14(test_df, val_tf),
                         batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

In [12]:
import numpy as np
from tqdm import tqdm
from sklearn.linear_model import RidgeClassifier
from sklearn.multiclass import OneVsRestClassifier

def loader_to_numpy(loader):
    X_list = []
    y_list = []

    for x, y in tqdm(loader):
        x = x.view(x.size(0), -1).cpu().numpy().astype(np.float16)
        y = y.cpu().numpy().astype(np.float32)

        X_list.append(x)
        y_list.append(y)

    X = np.concatenate(X_list, axis=0)
    Y = np.concatenate(y_list, axis=0)
    return X, Y

In [13]:
# Convert existing loaders to numpy arrays
X_train, y_train = loader_to_numpy(train_loader)
X_val, y_val = loader_to_numpy(val_loader)
X_test, y_test = loader_to_numpy(test_loader)

100%|██████████| 350/350 [05:12<00:00,  1.12it/s]


In [14]:
from sklearn.metrics import roc_auc_score
import numpy as np
import torch
from tqdm import tqdm

# Train Ridge Classifier
ridge = OneVsRestClassifier(RidgeClassifier(class_weight="balanced"))
ridge.fit(X_train, y_train)


OneVsRestClassifier(estimator=RidgeClassifier(class_weight='balanced'))

In [15]:
# Get decision scores
y_val_scores = ridge.decision_function(X_val)
y_test_scores = ridge.decision_function(X_test)

In [16]:
# AUROC
val_auc_per_class = roc_auc_score(y_val, y_val_scores, average=None)
val_mean_auc = roc_auc_score(y_val, y_val_scores, average="macro")


In [17]:
test_auc_per_class = roc_auc_score(y_test, y_test_scores, average=None)
test_mean_auc = roc_auc_score(y_test, y_test_scores, average="macro")



In [18]:
print("Validation Mean AUROC:", val_mean_auc)
for name, auc in zip(LABELS, val_auc_per_class):
    print(f"VAL  {name:18s}: {auc:.4f}")


Validation Mean AUROC: 0.6009853140412142
VAL  Atelectasis       : 0.5943
VAL  Cardiomegaly      : 0.6292
VAL  Effusion          : 0.6744
VAL  Infiltration      : 0.5799
VAL  Mass              : 0.5536
VAL  Nodule            : 0.5273
VAL  Pneumonia         : 0.5514
VAL  Pneumothorax      : 0.6036
VAL  Consolidation     : 0.6101
VAL  Edema             : 0.6612
VAL  Emphysema         : 0.5480
VAL  Fibrosis          : 0.5500
VAL  Pleural_Thickening: 0.5662
VAL  Hernia            : 0.7644


In [19]:
print("\nTest Mean AUROC:", test_mean_auc)
for name, auc in zip(LABELS, test_auc_per_class):
    print(f"TEST {name:18s}: {auc:.4f}")


Test Mean AUROC: 0.5829359538937784
TEST Atelectasis       : 0.6221
TEST Cardiomegaly      : 0.6158
TEST Effusion          : 0.6748
TEST Infiltration      : 0.5948
TEST Mass              : 0.5476
TEST Nodule            : 0.5186
TEST Pneumonia         : 0.5578
TEST Pneumothorax      : 0.5736
TEST Consolidation     : 0.6055
TEST Edema             : 0.6791
TEST Emphysema         : 0.5247
TEST Fibrosis          : 0.5673
TEST Pleural_Thickening: 0.5167
TEST Hernia            : 0.5629
